# Fama-French Multi-Factor Risk Decomposition

**Goal:** Decompose a stock's returns into market, size (SMB), and value (HML) factor exposures, and compare against the single-factor CAPM baseline.

This is a simplified version of the factor risk models (MSCI Barra, Axioma) used in real risk teams to answer: *where is this position's risk actually coming from?*

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from factor_data_loader import build_factor_dataset
from factor_model import run_capm, run_three_factor_model, compare_models

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Pull Data

Use the same ticker as your CAPM project for a direct before/after comparison.

In [ ]:
TICKER = 'AAPL'
START = '2015-01-01'
END = '2025-01-01'

df = build_factor_dataset(TICKER, START, END)
df.head()

## 2. Run Both Models Side-by-Side

In [ ]:
comparison, capm_result, ff3_result = compare_models(df)
comparison

In [ ]:
print(ff3_result['model'].summary())

## 3. R-Squared Improvement: CAPM vs 3-Factor

Does adding size and value factors meaningfully improve explanatory power, or is market beta doing almost all the work?

In [ ]:
fig, ax = plt.subplots()
models = ['CAPM\n(Market Only)', 'Fama-French\n3-Factor']
r2_values = [capm_result['r_squared'], ff3_result['r_squared']]

bars = ax.bar(models, r2_values, color=['steelblue', 'darkorange'])
ax.set_ylabel('R-squared')
ax.set_title(f'{TICKER}: Explanatory Power, CAPM vs 3-Factor Model')
ax.set_ylim(0, 1)

for bar, val in zip(bars, r2_values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.02, f'{val:.3f}', ha='center')

plt.tight_layout()
plt.savefig('../outputs/r_squared_comparison.png', dpi=150)
plt.show()

## 4. Factor Loadings — Which Risks Actually Matter?

Plot the 3-factor betas with their significance to see which factors are real drivers vs. noise.

In [ ]:
model = ff3_result['model']
factors = ['Mkt-RF', 'SMB', 'HML']
betas = [model.params[f] for f in factors]
pvalues = [model.pvalues[f] for f in factors]
colors = ['green' if p < 0.05 else 'lightgray' for p in pvalues]

fig, ax = plt.subplots()
bars = ax.bar(factors, betas, color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title(f'{TICKER}: Factor Loadings (green = significant at 5% level)')
ax.set_ylabel('Beta (Factor Loading)')

for bar, p in zip(bars, pvalues):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, height, f'p={p:.3f}',
            ha='center', va='bottom' if height >= 0 else 'top')

plt.tight_layout()
plt.savefig('../outputs/factor_loadings.png', dpi=150)
plt.show()

## 5. Residual Comparison

Do the 3-factor residuals look better behaved (less structure, smaller variance) than the CAPM-only residuals?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

axes[0].plot(df.index, capm_result['model'].resid, color='steelblue')
axes[0].set_title('CAPM Residuals')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

axes[1].plot(df.index, ff3_result['model'].resid, color='darkorange')
axes[1].set_title('3-Factor Residuals')
axes[1].axhline(0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../outputs/residual_comparison.png', dpi=150)
plt.show()

## 6. Extension: Single Stock vs. Diversified Portfolio

Factor models were built and validated on diversified portfolios, not individual stocks. A single name carries a lot of idiosyncratic, company-specific noise that dilutes the statistical power of factor loadings (e.g., a mega-cap's SMB loading can come out insignificant simply because single-stock noise swamps the signal).

This section re-runs the same 3-factor model on an equal-weighted portfolio of several stocks across different sectors, to see whether averaging away idiosyncratic risk produces tighter, more reliable factor estimates.

In [ ]:
from factor_data_loader import build_portfolio_factor_dataset

PORTFOLIO_TICKERS = ['AAPL', 'MSFT', 'JNJ', 'XOM', 'JPM', 'PG', 'KO', 'CAT']  # spread across sectors

df_portfolio = build_portfolio_factor_dataset(PORTFOLIO_TICKERS, START, END)
df_portfolio.head()

In [ ]:
portfolio_comparison, portfolio_capm, portfolio_ff3 = compare_models(df_portfolio)
portfolio_comparison

In [ ]:
print(portfolio_ff3['model'].summary())

### Side-by-Side: Single Stock vs. Portfolio

Compare standard errors and p-values directly. If the portfolio hypothesis is right, you should see:
- Smaller standard errors on each factor (tighter estimates)
- SMB more likely to reach significance (or at least a noticeably smaller p-value) once idiosyncratic noise is averaged out
- R² may actually be *similar or lower* than the single stock's — that's normal. Diversification reduces noise in the coefficient estimates, it doesn't necessarily raise the model's explanatory power over any one name.

In [ ]:
single_model = ff3_result['model']
portfolio_model = portfolio_ff3['model']

side_by_side = pd.DataFrame({
    f'{TICKER} (Single Stock) - Coef': single_model.params,
    f'{TICKER} (Single Stock) - Std Err': single_model.bse,
    f'{TICKER} (Single Stock) - p-value': single_model.pvalues,
    'Portfolio (8 stocks) - Coef': portfolio_model.params,
    'Portfolio (8 stocks) - Std Err': portfolio_model.bse,
    'Portfolio (8 stocks) - p-value': portfolio_model.pvalues,
})
side_by_side

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
factors = ['Mkt-RF', 'SMB', 'HML']

for ax, factor in zip(axes, factors):
    labels = [f'{TICKER}\n(single)', 'Portfolio\n(8 stocks)']
    coefs = [single_model.params[factor], portfolio_model.params[factor]]
    errs = [single_model.bse[factor], portfolio_model.bse[factor]]

    ax.bar(labels, coefs, yerr=errs, capsize=5, color=['steelblue', 'darkorange'])
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f'{factor} Loading \u00b1 Std Err')

plt.tight_layout()
plt.savefig('../outputs/single_vs_portfolio_factors.png', dpi=150)
plt.show()

### Extension Findings

_Fill in after running:_

- Did SMB's standard error shrink for the portfolio vs. the single stock? Did its p-value improve?
- Did any factor flip from insignificant to significant (or vice versa)?
- Does this support the idea that single-stock factor regressions are noisier and less reliable than portfolio-level ones?
- What would you tell a PM about the difference in confidence between a single-position factor exposure estimate and a portfolio-level one?

## 7. Findings & Risk Interpretation (Single Stock)

_Fill this in after reviewing your specific ticker's output. Prompts to answer:_

- How much did R² improve going from CAPM to the 3-factor model?
- Which factors (SMB, HML) were statistically significant? What does the sign tell you
  (small-cap vs large-cap tilt, value vs growth tilt)?
- Does the stock's factor profile match what you'd expect given the company (e.g., a large-cap
  growth tech name should show negative SMB and negative HML loadings)?
- What would you tell a portfolio manager about this position's *style* risk, not just its
  market risk?